# 01 · Datos y exploración (sección 2)

Pipeline: `LD2011_2014.txt` (kW cada 15 min) → kWh por hora → filtro de clientes completos 2012–2014 → corrección del cambio de horario → subconjunto fijo de clientes → panel largo `(unique_id, ds, y)` con covariables de calendario.

Todo el código vive en `src/data.py` y `src/eda.py`; este notebook solo lo ejecuta y muestra los resultados.

In [ ]:
import sys, json, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # avisos internos numpy/pandas por Timedelta
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from src import data as D, eda as E

pd.set_option("display.precision", 3)
FIGS = D.ROOT / "outputs" / "figs"
FIGS.mkdir(parents=True, exist_ok=True)

## 1. Construcción del panel

- **Agregación:** cada registro marca el final de su intervalo de 15 min (00:15 cubre 00:00–00:15), así que la hora 00:00 suma 00:15, 00:30, 00:45 y 01:00. Un registro en kW equivale a kW × 0.25 kWh.
- **Filtro:** un cliente es *completo* si tiene consumo el 2012-01-01 y no tiene rachas de más de 2 días seguidos en cero dentro de 2012–2014 (en el dataset, altas tardías y bajas aparecen como ceros).
- **Subconjunto:** 150 clientes elegidos con `seed=42` entre los completos, los mismos para todos los modelos (`data/clients.json`).

In [ ]:
panel = D.build_panel(force=True)   # genera data/hourly.parquet y data/clients.json
info = json.loads(D.CLIENTS_JSON.read_text())
E.panel_summary(panel)

In [ ]:
print(f"Clientes completos 2012-2014: {info['n_complete']}  ·  usados: {info['n_selected']}")
panel.head()

## 2. Cambio de horario

Portugal cambia de hora el último domingo de marzo y de octubre. Según la documentación del dataset, en esos días la hora 1:00–2:00 viene:

- **en marzo (día de 23 h):** en cero para todos los clientes → se **interpola linealmente** entre las 00:00 y las 02:00;
- **en octubre (día de 25 h):** con dos horas sumadas → se **divide entre 2**.

Así todas las series quedan con 24 valores por día, sin huecos ni picos artificiales, y el índice horario es regular (necesario para las ventanas de L = 168).

In [ ]:
march, october = D.dst_days(range(2012, 2015))
print("Marzo:  ", [str(d.date()) for d in march])
print("Octubre:", [str(d.date()) for d in october])

# Ejemplo: un cliente alrededor del cambio de marzo de 2013 (ya corregido)
uid = panel["unique_id"].iloc[0]
t = march[1]
panel.loc[(panel.unique_id == uid) & panel.ds.between(t, t + pd.Timedelta(hours=4)), ["ds", "y"]]

## 3. Perfil por hora del día y día de la semana

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
_, prof = E.plot_profiles(panel, ax=ax)
fig.tight_layout(); fig.savefig(FIGS / "perfil_hora_dia.png", dpi=150)

## 4. Feriados de Portugal, Navidad y Año Nuevo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
E.plot_daily_with_holidays(panel, ax=ax)
fig.tight_layout(); fig.savefig(FIGS / "diario_feriados.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
E.plot_christmas(panel, ax=ax)
fig.tight_layout(); fig.savefig(FIGS / "navidad.png", dpi=150)

# Cambio contra el promedio de las 4 semanas previas (mismo día de la semana)
E.christmas_effect(panel)

**Navidad y Año Nuevo.** El 25 de diciembre el consumo cae entre **29 y 33 %** respecto al promedio de las 4 semanas previas del mismo día de la semana, y el 1 de enero cae **~22 %**: el perfil de esos días se parece al de un domingo muy bajo, con el pico de la tarde casi desaparecido. Nochebuena y Nochevieja solo bajan **2–8 %**, y la caída se concentra en la tarde-noche. Entre Navidad y Año Nuevo los días hábiles vuelven casi a la normalidad. Para los modelos esto es un cambio de régimen: la semana del 22 al 28 de diciembre de prueba contiene el 25, que ningún rezago de 24 h ni de 168 h anticipa; solo la covariable de feriado puede avisarlo.

## 5. Escalas entre clientes y normalización

In [ ]:
summary, means = E.scale_stats(panel)
display(summary.to_frame("kWh/h promedio"))
fig, ax = plt.subplots(figsize=(6, 3.5))
E.plot_scales(means, ax=ax)
fig.tight_layout(); fig.savefig(FIGS / "escalas.png", dpi=150)

**Normalización elegida: z-score por serie con estadísticos de entrenamiento** (`SeriesScaler` en `src/data.py`).

- Las escalas difieren en varios órdenes de magnitud (ver `max/min`). Con una normalización **global**, la pérdida quedaría dominada por los clientes grandes y los pequeños serían casi cero para la red.
- **Por serie** pone a todos los clientes en la misma escala y permite entrenar un modelo global. Media y desviación se calculan **solo con datos hasta 2014-06-30**, así que validación y prueba no influyen en la escala (evita la fuga de escalar con toda la serie).
- **Por ventana** (RevIN) se deja como variante en DLinear/NLinear: ayuda ante cambios de nivel, pero se compara en las iteraciones de la sección 5.2.
- Las métricas siempre se calculan en kWh, después de `SeriesScaler.inverse`.

In [ ]:
scaler = D.SeriesScaler().fit(panel)
z = scaler.transform(panel)
m = D.split_masks(z["ds"])
z[m["train"]].groupby("unique_id", observed=True)["y"].agg(["mean", "std"]).describe().loc[["mean", "min", "max"]]

## 6. Covariables de calendario (conocidas a futuro)

| Covariable | Codificación | Motivo |
|---|---|---|
| Hora (0–23) | seno/coseno con periodo 24 | Es cíclica: las 23 h quedan cerca de las 0 h; 2 columnas en lugar de 24 |
| Día de la semana | seno/coseno con periodo 7 | Igual, domingo cerca del lunes |
| Mes | seno/coseno con periodo 12 | Estacionalidad anual |
| Feriado | binaria (`holidays.Portugal`) | Cambia el perfil completo del día; la librería respeta los feriados suspendidos en 2013–2015 |

Todas dependen solo de la fecha, así que se pueden calcular para las 24 horas del horizonte en el momento de emitir el pronóstico.

In [ ]:
panel[D.CALENDAR_COLS].describe().loc[["min", "max", "mean"]]

In [ ]:
# Feriados del periodo
D.pt_holidays(range(2012, 2015)).to_series().dt.strftime("%Y-%m-%d %a").to_frame("feriado").T

## 7. Auditoría de fugas (parte A)

**Escalar con toda la serie.** `SeriesScaler` se ajusta solo con `ds <= 2014-06-30`. La prueba de abajo comprueba que cambiar los datos de validación y prueba no cambia los estadísticos.

In [ ]:
import numpy as np
perturbado = panel.copy()
perturbado.loc[perturbado["ds"] > D.TRAIN_END, "y"] *= 100   # destruimos val + test
s1 = D.SeriesScaler().fit(panel).stats
s2 = D.SeriesScaler().fit(perturbado).stats
assert np.allclose(s1, s2), "los estadísticos dependen de datos fuera de entrenamiento"
print("OK: la normalización solo usa datos de entrenamiento")

**Covariables mal alineadas.** Se prueba sobre el `WindowDataset` (sección 8): la entrada termina en `t0 − 1h`, cambiar el futuro no cambia `x`, y el calendario del horizonte coincide hora por hora con el que se calcula solo a partir de las fechas objetivo. Las pruebas están al final de la sección 8.

## 8. Dataset de ventanas (PyTorch)

`WindowDataset` (`src/dataset.py`) arma ventanas de todas las series con un índice perezoso `(serie, t0)`, donde `t0` es la primera hora del horizonte:

| Tensor | Forma | Contenido |
|---|---|---|
| `x` | `(batch, L, 8)` | y normalizada + 7 covariables de calendario, horas `t0−L … t0−1` |
| `x_fut` (opcional) | `(batch, H, 7)` | calendario de las horas `t0 … t0+H−1` (para seq2seq) |
| `y` | `(batch, H)` | objetivo normalizado, horas `t0 … t0+H−1` |

Una ventana pertenece a un split si **todas sus horas objetivo** están dentro del split; la entrada puede venir del bloque anterior porque es historia conocida al emitir el pronóstico. `batches()` arma los batches directo en la GPU (una época de entrenamiento completa tarda ~1 s).

In [ ]:
from src.dataset import WindowDataset, count_windows

counts = count_windows(z, L=168, H=24, stride=1)
counts

Por serie: entrenamiento tiene 21,888 horas → 21,888 − 168 − 24 + 1 = **21,697** ventanas; validación y prueba tienen 2,208 horas cada uno → 2,208 − 24 + 1 = **2,185** ventanas (la entrada sale del bloque anterior). En el backtest de prueba solo se usan los orígenes de las 00:00: **92 × 150 = 13,800** ventanas.

In [ ]:
tr = WindowDataset(z, split="train")
x, y = tr[0]
print("x:", tuple(x.shape), " y:", tuple(y.shape))

te = WindowDataset(z, split="test", origin_hour=0, with_future=True)
x, x_fut, y = te[0]
print("test diario:", len(te), "ventanas  ·  x:", tuple(x.shape), " x_fut:", tuple(x_fut.shape), " y:", tuple(y.shape))
te.window_info().head(3)

In [ ]:
# Batches con DataLoader estándar y con el iterador rápido en GPU
from torch.utils.data import DataLoader
import torch
xb, yb = next(iter(DataLoader(tr, batch_size=512, shuffle=True)))
print("DataLoader:", tuple(xb.shape), tuple(yb.shape))
device = "cuda" if torch.cuda.is_available() else "cpu"
xb, yb = next(tr.batches(512, shuffle=True, device=device, seed=0))
print("batches():", tuple(xb.shape), tuple(yb.shape), xb.device)

### Auditoría: covariables mal alineadas

Cada entrada de la red debe poder calcularse a las 00:00 del día pronosticado. Tres pruebas:

1. **La entrada termina a las 23:00 del día anterior** (`last_input = origin − 1h`).
2. **Cambiar el futuro no cambia `x`:** se multiplican por 100 todos los valores desde `t0` y la entrada queda idéntica.
3. **El calendario del horizonte está alineado:** `x_fut[h]` es exactamente `calendar_features(t0 + h)`, calculado solo con la fecha. Una versión desplazada una hora (el error típico de usar `shift`) no pasa la prueba.

In [ ]:
info = te.window_info()
assert (info["origin"].dt.hour == 0).all()
assert (info["last_input"] == info["origin"] - pd.Timedelta(hours=1)).all()
assert (info["last_input"].dt.hour == 23).all()
print("1. OK: la última hora de entrada es las 23:00 del día anterior")

i = 10
w = te.window_info([i]).iloc[0]
z_mod = z.copy()
futuro = (z_mod["unique_id"] == w.unique_id) & (z_mod["ds"] >= w.origin)
z_mod.loc[futuro, "y"] *= 100
te_mod = WindowDataset(z_mod, split="test", origin_hour=0, with_future=True)
assert torch.equal(te[i][0], te_mod[i][0]), "x depende de datos futuros"
assert not torch.equal(te[i][2], te_mod[i][2])
print("2. OK: modificar el futuro no cambia la entrada x (sí cambia y, como debe)")

esperado = torch.tensor(D.calendar_features(te.target_times(i)).to_numpy())
assert torch.allclose(te[i][1], esperado)
desplazado = torch.tensor(D.calendar_features(te.target_times(i) - pd.Timedelta(hours=1)).to_numpy())
assert not torch.allclose(te[i][1], desplazado)
print("3. OK: x_fut[h] = calendario de t0 + h; una versión corrida 1 h no coincide")